# Clase 7 — NLP II: sentimiento, redes recurrentes, LSTM y generación de texto

**Objetivos de la clase**

- Entrenar un modelo de análisis de sentimiento sobre texto real
- Explicar qué resuelven las redes recurrentes (RNN, Recurrent Neural Network) frente a una red densa para datos secuenciales, y qué aporta LSTM (Long Short-Term Memory) frente a una RNN simple
- Generar texto nuevo (poesía) a partir de un modelo entrenado con LSTM
- Reutilizar embeddings preentrenados con TensorFlow Hub para clasificar texto

Este notebook es la guía de la clase en vivo: código funcionando, recortado en número de épocas para que puedas ejecutarlo entero durante la sesión. El manual (capítulo 09) desarrolla cada concepto con más detalle.

In [ ]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)

## 1. Clasificador de sentimiento con embeddings

Partimos del dataset de titulares sarcásticos/no sarcásticos (ya lo tokenizaste en la clase anterior). La arquitectura: **Embedding** (aprende un vector por palabra) → **GlobalAveragePooling1D** (promedia esos vectores por frase) → dos capas **Dense** de clasificación.

In [ ]:
import json
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/sarcasm.json \
    -O /tmp/sarcasm.json

In [ ]:
vocab_size = 10000
embedding_dim = 16
max_length = 100
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"
training_size = 20000

with open("/tmp/sarcasm.json", 'r') as f:
    datastore = json.load(f)

sentences = [item['headline'] for item in datastore]
labels = [item['is_sarcastic'] for item in datastore]

training_sentences = sentences[0:training_size]
testing_sentences = sentences[training_size:]
training_labels = np.array(labels[0:training_size])
testing_labels = np.array(labels[training_size:])

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(training_sentences)

training_sequences = tokenizer.texts_to_sequences(training_sentences)
training_padded = pad_sequences(training_sequences, maxlen=max_length,
                                 padding=padding_type, truncating=trunc_type)

testing_sequences = tokenizer.texts_to_sequences(testing_sentences)
testing_padded = pad_sequences(testing_sequences, maxlen=max_length,
                                padding=padding_type, truncating=trunc_type)

print(training_padded.shape, testing_padded.shape)

In [ ]:
sentiment_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

sentiment_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
sentiment_model.summary()

In [ ]:
# Recortamos a 10 épocas para que la celda se ejecute rápido en clase
# (en el manual se entrena con 30 épocas y se observa más sobreajuste)
history = sentiment_model.fit(
    training_padded, training_labels,
    epochs=10,
    validation_data=(testing_padded, testing_labels),
    verbose=2
)

In [ ]:
import matplotlib.pyplot as plt

def plot_graphs(history, string):
    plt.plot(history.history[string])
    plt.plot(history.history['val_' + string])
    plt.xlabel("Epochs")
    plt.ylabel(string)
    plt.legend([string, 'val_' + string])
    plt.show()

plot_graphs(history, "accuracy")
plot_graphs(history, "loss")

In [ ]:
nuevas_frases = [
    "granny starting to fear spiders in the garden might be real",
    "game of thrones season finale showing this sunday night"
]
secuencias = tokenizer.texts_to_sequences(nuevas_frases)
padded = pad_sequences(secuencias, maxlen=max_length, padding=padding_type, truncating=trunc_type)
print(sentiment_model.predict(padded))

## 2. Reutilizar embeddings preentrenados con TensorFlow Hub

En vez de entrenar un embedding desde cero, usamos uno ya entrenado por Google (`nnlm-en-dim50`) para clasificar reseñas de películas del dataset IMDB (Internet Movie Database) como positivas o negativas. La capa de TensorFlow Hub recibe directamente el texto, sin tokenizar nosotros mismos.

In [ ]:
!pip install -q tensorflow-hub tensorflow-datasets

In [ ]:
import tensorflow_hub as hub
import tensorflow_datasets as tfds

# 60% train / 40% validación, más el split de test oficial
train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]', 'train[60%:]', 'test'),
    as_supervised=True
)

train_examples_batch, train_labels_batch = next(iter(train_data.batch(3)))
train_examples_batch

In [ ]:
embedding_url = "https://tfhub.dev/google/nnlm-en-dim50/2"
hub_layer = hub.KerasLayer(embedding_url, input_shape=[], dtype=tf.string, trainable=True)

# Comprobación rápida: cualquier frase se convierte en un vector de 50 valores
hub_layer(train_examples_batch[:3])

In [ ]:
hub_model = tf.keras.Sequential([
    hub_layer,
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])

hub_model.compile(optimizer='adam',
                   loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                   metrics=['accuracy'])
hub_model.summary()

In [ ]:
# Recortamos a 3 épocas para la clase (el tutorial original usa 10)
hub_history = hub_model.fit(
    train_data.shuffle(10000).batch(512),
    epochs=3,
    validation_data=validation_data.batch(512),
    verbose=1
)

results = hub_model.evaluate(test_data.batch(512), verbose=2)
for name, value in zip(hub_model.metrics_names, results):
    print("%s: %.3f" % (name, value))

## 3. Por qué una red densa no basta para datos secuenciales: la RNN

`GlobalAveragePooling1D` promedia los vectores de todas las palabras: le da igual el orden. Eso funciona para sentimiento general, pero no sirve si queremos **predecir la siguiente palabra** de una frase, porque ahí el orden es la información.

Una red neuronal recurrente (RNN) procesa la secuencia palabra a palabra, y cada paso recibe además el resultado del paso anterior (su "memoria"). Es la misma idea que la sucesión de Fibonacci: cada número arrastra información de los anteriores.

**Limitación de una RNN simple:** ese rastro se diluye con la distancia (el problema del gradiente que desaparece). En "I lived in Ireland, so I learned how to speak ___", la pista ("Ireland") está lejos de la palabra a predecir ("Gaelic"), y una RNN simple tiende a perderla.

## 4. LSTM: memoria a largo-corto plazo

LSTM añade un **estado de celda** que actúa como cinta transportadora de información a largo plazo, regulado por tres puertas (gates): de entrada, de olvido y de salida. Esto le permite conservar información relevante aunque esté lejos en la secuencia.

Vamos a comprobarlo de forma práctica: sustituimos `GlobalAveragePooling1D` por una capa `LSTM` en el clasificador de sarcasmo y comparamos.

In [ ]:
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

lstm_history = lstm_model.fit(
    training_padded, training_labels,
    epochs=10,
    validation_data=(testing_padded, testing_labels),
    verbose=2
)

Compara la columna `val_accuracy` de este entrenamiento con la del modelo con `GlobalAveragePooling1D` de la sección 1. No hay una versión "siempre mejor": una LSTM tiene más parámetros y tarda más por época; si el problema no depende mucho del orden de las palabras, puede no compensar.

**Detalles que usarás en el código de LSTM:**
- `Bidirectional(LSTM(n))` procesa la frase en ambas direcciones y combina los resultados
- Si apilas varias capas LSTM, todas menos la última necesitan `return_sequences=True` para pasar la secuencia completa a la siguiente capa

## 5. Generar texto nuevo (poesía) con LSTM

Cambiamos de tarea: en vez de clasificar, vamos a **predecir la siguiente palabra**. Entrenamos con letras de canciones tradicionales irlandesas, generamos todos los n-gramas de cada línea y enseñamos al modelo a predecir la última palabra de cada n-grama a partir de las anteriores.

In [ ]:
!wget --no-check-certificate \
    https://storage.googleapis.com/learning-datasets/irish-lyrics-eof.txt \
    -O /tmp/irish-lyrics-eof.txt

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

poem_tokenizer = Tokenizer()
data = open('/tmp/irish-lyrics-eof.txt').read()
corpus = data.lower().split("\n")

poem_tokenizer.fit_on_texts(corpus)
total_words = len(poem_tokenizer.word_index) + 1
print("Palabras en el vocabulario:", total_words)

In [ ]:
input_sequences = []
for line in corpus:
    token_list = poem_tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

xs, labels_poem = input_sequences[:, :-1], input_sequences[:, -1]
ys = tf.keras.utils.to_categorical(labels_poem, num_classes=total_words)

print("Forma de xs:", xs.shape, "- Forma de ys:", ys.shape)

In [ ]:
# Hiperparámetros reducidos (embedding y unidades LSTM más pequeños, menos
# épocas) para que el entrenamiento sea rápido en clase. El tutorial original
# usa 100 unidades de embedding, 150 unidades LSTM y 100 épocas.
poem_model = Sequential([
    Embedding(total_words, 64, input_length=max_sequence_len - 1),
    Bidirectional(LSTM(64)),
    Dense(total_words, activation='softmax')
])

poem_model.compile(loss='categorical_crossentropy',
                    optimizer=Adam(learning_rate=0.01),
                    metrics=['accuracy'])
poem_model.summary()

In [ ]:
poem_history = poem_model.fit(xs, ys, epochs=30, verbose=1)

In [ ]:
plot_graphs(poem_history, 'accuracy')

In [ ]:
def generar_texto(seed_text, next_words, model, tokenizer, max_sequence_len):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

seed_text = "I've got a bad feeling about this"
print(generar_texto(seed_text, 20, poem_model, poem_tokenizer, max_sequence_len))

## Resumen de la clase

- Un clasificador de sentimiento sencillo: Embedding + GlobalAveragePooling1D + Dense
- TensorFlow Hub te da embeddings ya entrenados como primera capa, sin tokenizar tú mismo (aprendizaje por transferencia)
- Una RNN añade memoria de lo que ya ha visto; una red densa no
- LSTM resuelve el problema del gradiente que desaparece con un estado de celda y tres puertas (entrada, olvido, salida)
- Generar texto es predecir la siguiente palabra, entrenando con n-gramas y etiquetas one-hot, y repitiendo la predicción sobre una semilla (seed)

Sigue con el notebook de práctica de esta clase para aplicar estos cambios tú mismo.